# Baseline CNN Model
### Chest X-Ray Classification: Normal vs Pneumonia vs COVID-19
This notebook builds a simple CNN from scratch as a baseline model before moving to transfer learning.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

/opt/anaconda3/envs/pneumonia-detection/lib/python3.11/site-packages/torchvision/io/image.py:14: UserWarning: Failed to load image Python extension: 'dlopen(/opt/anaconda3/envs/pneumonia-detection/lib/python3.11/site-packages/torchvision/image.so, 0x0006): Library not loaded: @rpath/libjpeg.9.dylib
  Referenced from: <EB3FF92A-5EB1-3EE8-AF8B-5923C1265422> /opt/anaconda3/envs/pneumonia-detection/lib/python3.11/site-packages/torchvision/image.so
  Reason: tried: '/opt/anaconda3/envs/pneumonia-detection/lib/python3.11/site-packages/torchvision/../../../libjpeg.9.dylib' (no such file), '/opt/anaconda3/envs/pneumonia-detection/lib/python3.11/site-packages/torchvision/../../../libjpeg.9.dylib' (no such file), '/opt/anaconda3/envs/pneumonia-detection/lib/python3.11/lib-dynload/../../libjpeg.9.dylib' (no such file), '/opt/anaconda3/envs/pneumonia-detection/bin/../lib/libjpeg.9.dylib' (no such file)'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warnin

In [2]:
# Use Apple GPU if available, otherwise CPU
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)

Using device: mps


In [3]:
# Dataset paths
data_dir = Path('../data/Dataset')
train_dir = data_dir / 'Train_Validation'
test_dir = data_dir / 'Test'
classes = ['COVID', 'Normal', 'Pneumonia']

# Preprocessing pipeline - resize, convert to tensor, normalise
transform = transforms.Compose([
    transforms.Resize((224, 224)),        # resize to ResNet standard
    transforms.ToTensor(),                 # convert to tensor, scale 0-1
    transforms.Normalize(mean=[0.485, 0.456, 0.406],   # ImageNet mean
                        std=[0.229, 0.224, 0.225])      # ImageNet std
])

In [4]:
class XRayDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        self.data_dir = Path(data_dir)
        self.transform = transform
        self.image_paths = []
        self.labels = []
        
        # Loop through each class folder and collect image paths and labels
        for label, cls in enumerate(classes):
            class_dir = self.data_dir / cls
            for img_path in class_dir.glob('*'):
                self.image_paths.append(img_path)
                self.labels.append(label)
    
    def __len__(self):
        # Return total number of images
        return len(self.image_paths)
    
    def __getitem__(self, index):
        # Load, convert to RGB and preprocess one image
        img_path = self.image_paths[index]
        image = Image.open(img_path).convert('RGB')
        image = self.transform(image)
        label = self.labels[index]
        return image, label

In [5]:
# Create full dataset
full_dataset = XRayDataset(train_dir, transform=transform)
test_dataset = XRayDataset(test_dir, transform=transform)

# Split into 80% train, 20% validation BEFORE any training
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_split, val_split = random_split(full_dataset, [train_size, val_size])

# Create DataLoaders
train_loader = DataLoader(train_split, batch_size=32, shuffle=True)
val_loader = DataLoader(val_split, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("Training samples:", train_size)
print("Validation samples:", val_size)
print("Test samples:", len(test_dataset))

Training samples: 2308
Validation samples: 578
Test samples: 341


In [9]:
# Test forward pass with one batch
images, labels = next(iter(train_loader))
output = model(images)
print("Input shape:", images.shape)
print("Output shape:", output.shape)

RuntimeError: Mismatched Tensor types in NNPack convolutionOutput

In [6]:
class BaseCNN(nn.Module):
    def __init__(self):
        super(BaseCNN, self).__init__()
        
        # First convolutional block
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Second convolutional block
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        
        # Third convolutional block
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        
        # Fully connected classifier
        self.fc1 = nn.Linear(128 * 28 * 28, 512)
        self.fc2 = nn.Linear(512, 3)  # 3 output classes
        
        # Dropout for regularisation
        self.dropout = nn.Dropout(0.5)
        
        # ReLU activation
        self.relu = nn.ReLU()

    def forward(self, x):
        # First conv block: conv -> relu -> pool
        x = self.pool(self.relu(self.conv1(x)))
        
        # Second conv block
        x = self.pool(self.relu(self.conv2(x)))
        
        # Third conv block
        x = self.pool(self.relu(self.conv3(x)))
        
        # Flatten for fully connected layers
        x = x.view(x.size(0), -1)
        
        # Fully connected layers with dropout
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.fc2(x)
        
        return x

# Create model and move to device
model = BaseCNN().to(device)
print(model)

BaseCNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=100352, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=3, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (relu): ReLU()
)


In [7]:
# Loss function for multi-class classification
criterion = nn.CrossEntropyLoss()

# Adam optimiser with learning rate 0.001
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [8]:
num_epochs = 10

for epoch in range(num_epochs):
    # Training phase
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in train_loader:
        # Move data to device
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()        # clear gradients
        outputs = model(images)      # forward pass
        loss = criterion(outputs, labels)  # calculate loss
        loss.backward()              # backward pass
        optimizer.step()             # update weights
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    
    train_loss = running_loss / len(train_loader)
    train_acc = 100 * correct / total
    
    # Validation phase
    model.eval()
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
    
    val_acc = 100 * val_correct / val_total
    print(f"Epoch {epoch+1}/{num_epochs} - Loss: {train_loss:.4f} - Train Acc: {train_acc:.2f}% - Val Acc: {val_acc:.2f}%")

Epoch 1/10 - Loss: 0.9748 - Train Acc: 55.68% - Val Acc: 68.17%
Epoch 2/10 - Loss: 0.6768 - Train Acc: 72.92% - Val Acc: 70.93%
Epoch 3/10 - Loss: 0.5570 - Train Acc: 78.47% - Val Acc: 73.53%
Epoch 4/10 - Loss: 0.4414 - Train Acc: 82.58% - Val Acc: 79.41%
Epoch 5/10 - Loss: 0.3644 - Train Acc: 85.70% - Val Acc: 78.03%
Epoch 6/10 - Loss: 0.2766 - Train Acc: 89.34% - Val Acc: 76.99%
Epoch 7/10 - Loss: 0.2273 - Train Acc: 91.03% - Val Acc: 76.64%
Epoch 8/10 - Loss: 0.1988 - Train Acc: 92.42% - Val Acc: 79.76%
Epoch 9/10 - Loss: 0.1300 - Train Acc: 95.19% - Val Acc: 80.28%
Epoch 10/10 - Loss: 0.0999 - Train Acc: 96.27% - Val Acc: 77.34%


In [ ]:
# Save the trained baseline model
torch.save(model.state_dict(), '../models/baseline_cnn.pth')
print("Model saved successfully")